# Weekly Challenge 02 – House Hunters 🏠
*Tabular regression · difficulty ★★☆☆ · ~1 evening*

An estate agency in California wants instant price estimates. For each **census block group** you get 8 features
(median income, house age, average rooms, population, latitude, longitude, …) and must predict the
**median house value** (in units of $100 000).

The current tool is a plain linear regression with an RMSE of about 0.75 – i.e. typical errors of ~$75 000.
Your job: **get the test RMSE down to 0.48 or lower.**

| | |
|---|---|
| **Data** | `sklearn.datasets.fetch_california_housing` – 20 640 rows, 8 numeric features (downloads ≈ 400 KB once, then cached) |
| **Split** | random 80/20, `random_state=42` (fixed) |
| **Metric** | RMSE on the test set (lower is better) |
| **Baseline** | `LinearRegression` → **0.746** |
| **Target** | **≤ 0.48** |

Theory: [[Linear Regression]] · [[Feature Engineering]] · [[Decision Trees]] · [[Ensemble Methods]]

**How it works**
1. Read the story, run the setup and the baseline.
2. Build your model in the ✏️ *Your turn* cell (it starts as a copy of the baseline).
3. Run the 🏁 *Evaluate* cell – it prints ✅ if you beat the target, ❌ otherwise.
4. Stuck? Open the 💡 hints one at a time. Done (or truly stuck)? Open the 🏆 solution walkthrough.
5. Log your best score in [[Weekly Challenges]].

**Rules** – make every modelling decision with the training data only (cross-validation or a validation split).
The test set is touched exactly once, by the evaluate cell. Peeking at it to tune is cheating yourself.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
warnings.filterwarnings("ignore")

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def report(name, score, baseline, target, higher_is_better=True):
    beat = score >= target if higher_is_better else score <= target
    arrow = "≥" if higher_is_better else "≤"
    print(f"{name}: {score:.4f}   (baseline {baseline:.4f}, target {arrow} {target})")
    print(("✅ Target beaten – log it in Weekly Challenges!" if beat
           else "❌ Not yet – keep going (or open a hint)."))

## 1 · Load the data (fixed split)

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)
X_train.head()

## 2 · Have a look
Price on a map. Is the relation between *location* and price something a straight line can capture?

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X_train.Longitude, X_train.Latitude, c=y_train, s=2, cmap="viridis")
plt.colorbar(label="median house value ($100k)"); plt.xlabel("longitude"); plt.ylabel("latitude")
plt.title("Coastal = expensive"); plt.show()

## 3 · Baseline

In [ ]:
from sklearn.linear_model import LinearRegression

baseline = LinearRegression().fit(X_train, y_train)
BASELINE = rmse(y_test, baseline.predict(X_test))
TARGET = 0.48
print(f"baseline test RMSE: {BASELINE:.3f}")

## 4 · ✏️ Your turn
Define `my_model` – any scikit-learn estimator or pipeline. Compare ideas with cross-validated RMSE on the training set.

In [ ]:
# ✏️ TODO: replace this with your own model
my_model = LinearRegression()

cv = -cross_val_score(my_model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
print(f"5-fold CV RMSE on train: {cv.mean():.3f} ± {cv.std():.3f}")

## 5 · 🏁 Evaluate

In [ ]:
my_model.fit(X_train, y_train)
score = rmse(y_test, my_model.predict(X_test))
report("test RMSE", score, BASELINE, TARGET, higher_is_better=False)

<details><summary>💡 Hint 1</summary>

Look at the map again: price depends on *where* you are in a very non-linear way (coast, Bay Area, LA). A linear model can only tilt a plane over the map.
</details>

<details><summary>💡 Hint 2</summary>

Tree-based models carve the feature space into boxes – perfect for "lat between A and B *and* lon between C and D". Try a random forest or gradient boosting.
</details>

<details><summary>💡 Hint 3</summary>

`HistGradientBoostingRegressor` is fast on 16 000 rows and is close to the target with default settings. Then tune `max_iter`, `max_leaf_nodes` and `learning_rate` with CV.
</details>

<details>
<summary>🏆 <b>Solution walkthrough</b> (click to open – try first!)</summary>

**Why the baseline is weak.** Linear regression assumes each feature adds a fixed amount to the price. But the
effect of latitude depends on longitude (the coast runs diagonally), income has diminishing effects, and
the target is capped at 5.0. None of this fits a single plane.

**Step 1 – trees.** Gradient-boosted trees learn interactions and thresholds automatically. A default
`HistGradientBoostingRegressor(random_state=0)` already reaches ≈ 0.462 test RMSE.

**Step 2 – tune with CV.** More boosting rounds and bigger trees help a little: `max_iter=600, max_leaf_nodes=63`
gives ≈ 0.463 in 5-fold CV and **0.451 on the test set** – about 40 % lower error than the baseline.

**Feature engineering alternative.** If you want to stay linear: add rooms per household, a rotated coordinate
(lat + lon) and distances to San Francisco/Los Angeles, then expand every feature with `SplineTransformer` and fit
`RidgeCV`. That reaches ≈ 0.56–0.61 depending on the number of knots – a big gain over 0.746, but still well short of
boosting with zero feature engineering. A nice demonstration of why boosting is the default for tabular data.

**Lesson.** On tabular data with interactions, start with gradient boosting; spend your feature-engineering effort where the model can't help itself.

```python
from sklearn.ensemble import HistGradientBoostingRegressor

my_model = HistGradientBoostingRegressor(max_iter=600, max_leaf_nodes=63, learning_rate=0.1, random_state=0)
cv = -cross_val_score(my_model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
print(f"5-fold CV RMSE on train: {cv.mean():.3f} ± {cv.std():.3f}")
```
</details>